# Costa del Sol Flood Risk - Notebook 2: Indicators & Analysis

## 0. Setup

In [1]:
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask as rio_mask

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

admin_municipios = gpd.read_file(PROCESSED_DIR / "admin_municipios_25830.gpkg")
admin_municipios["area_municipio_km2"] = admin_municipios.geometry.area / 1e6
print(f"{len(admin_municipios)} municipios de estudio cargados")
admin_municipios[["Mun_Code", "Mun_Name", "area_municipio_km2"]].head()

62 municipios de estudio cargados


,Mun_Code,Mun_Name,area_municipio_km2
0,11004,Algeciras,88.001424
1,11008,Los Barrios,330.865635
2,11013,Castellar de la Frontera,180.579288
3,11021,Jimena de la Frontera,298.538629
4,11022,La Línea de la Concepción,26.674262


## 0. Seleccion de municipios para el grid de detalle

Antes de decidir que municipios llevan el grid de zoom (250-500m), generamos tres listados
independientes de los 62 municipios - completos, sin autoseleccionar un top-N. La idea es
verlos enteros y elegir a mano, no que el notebook decida solo:

1. Superficie inundada por municipio, para los 4 periodos de retorno (T10/T50/T100/T500) -
   area absoluta y % del termino municipal. T100 como referencia principal.
2. Uso de suelo dentro de la zona inundada (no del municipio entero).
3. EUR/m2 de vivienda, como eje independiente de valor economico.

Se descarta el tamano de cuenca como proxy de superficie inundable - ya tenemos el dato real
(listado 1), usar el proxy en su lugar seria peor evidencia.

### 0.1 Superficie inundada por municipio y periodo de retorno

Los 4 gpkg de SNCZI (`snczi_t10/t50/t100/t500_25830.gpkg`) estan recortados por la extension
**hidrologica** (la cuenca completa), no por el limite administrativo de los 62 municipios -
por eso se intersecta con `admin_municipios` en vez de asumir que ya esta contenido.

Se guarda tambien la geometria inundada por municipio (union de los trozos que caigan dentro,
por periodo) - se reutiliza en 0.2 para enmascarar los rasters de uso de suelo.

In [2]:
PERIODOS = ["t10", "t50", "t100", "t500"]

flood_by_mun = admin_municipios[["Mun_Code", "Mun_Name", "area_municipio_km2"]].copy()
flood_geoms = {}  # (Mun_Code, periodo) -> geometria inundada dentro del municipio - para 0.2

for periodo in PERIODOS:
    snczi = gpd.read_file(PROCESSED_DIR / f"snczi_{periodo}_25830.gpkg")

    inter = gpd.overlay(
        admin_municipios[["Mun_Code", "geometry"]],
        snczi[["geometry"]],
        how="intersection",
    )
    inter["area_km2"] = inter.geometry.area / 1e6

    area_by_mun = inter.groupby("Mun_Code")["area_km2"].sum()
    union_by_mun = inter.dissolve(by="Mun_Code").geometry

    for mc, geom in union_by_mun.items():
        flood_geoms[(mc, periodo)] = geom

    flood_by_mun[f"area_inundada_{periodo}_km2"] = (
        flood_by_mun["Mun_Code"].map(area_by_mun).fillna(0)
    )
    flood_by_mun[f"pct_inundado_{periodo}"] = (
        flood_by_mun[f"area_inundada_{periodo}_km2"] / flood_by_mun["area_municipio_km2"] * 100
    )
    print(f"{periodo}: {(flood_by_mun[f'area_inundada_{periodo}_km2'] > 0).sum()} municipios con algo de superficie inundada")

flood_by_mun = flood_by_mun.sort_values("pct_inundado_t100", ascending=False).reset_index(drop=True)
flood_by_mun.to_csv(PROCESSED_DIR / "flooded_area_by_municipio.csv", index=False)

print(f"\n{flood_by_mun.shape[0]} municipios, ordenados por % inundado en T100 (de mayor a menor)")
pd.set_option("display.max_rows", None)
flood_by_mun

t10: 37 municipios con algo de superficie inundada
t50: 37 municipios con algo de superficie inundada
t100: 37 municipios con algo de superficie inundada
t500: 37 municipios con algo de superficie inundada

62 municipios, ordenados por % inundado en T100 (de mayor a menor)


,Mun_Code,Mun_Name,area_municipio_km2,area_inundada_t10_km2,pct_inundado_t10,area_inundada_t50_km2,pct_inundado_t50,area_inundada_t100_km2,pct_inundado_t100,area_inundada_t500_km2,pct_inundado_t500
0,29038,Cártama,105.097959,1.033124e+01,9.830105e+00,1.262583e+01,1.201339e+01,1.349464e+01,1.284006e+01,1.452226e+01,1.381783e+01
1,29080,Pizarra,63.848656,4.544800e+00,7.118083e+00,6.864449e+00,1.075113e+01,7.487019e+00,1.172620e+01,8.273123e+00,1.295739e+01
2,29054,Fuengirola,10.305779,4.349117e-01,4.220076e+00,9.760995e-01,9.471380e+00,1.039753e+00,1.008903e+01,1.150792e+00,1.116647e+01
3,29094,Vélez-Málaga,157.833838,1.055601e+01,6.688052e+00,1.221208e+01,7.737303e+00,1.290050e+01,8.173469e+00,1.417596e+01,8.981572e+00
4,29026,Benamargosa,12.112582,7.932722e-01,6.549159e+00,8.373297e-01,6.912892e+00,8.608501e-01,7.107073e+00,9.124207e-01,7.532834e+00
5,11033,San Roque,139.781297,4.780433e+00,3.419938e+00,7.887376e+00,5.642655e+00,8.392951e+00,6.004344e+00,1.078058e+01,7.712463e+00
6,29005,Algarrobo,9.730080,2.957934e-01,3.039989e+00,3.758183e-01,3.862438e+00,4.744534e-01,4.876151e+00,7.503562e-01,7.711716e+00
7,29007,Alhaurín de la Torre,82.674712,2.700869e+00,3.266862e+00,3.732903e+00,4.515170e+00,4.000381e+00,4.838700e+00,4.285891e+00,5.184041e+00
8,11903,San Martín del Tesorillo,48.197912,6.834556e-01,1.418019e+00,2.143320e+00,4.446914e+00,2.300567e+00,4.773166e+00,2.522389e+00,5.233399e+00
9,29067,Málaga,395.388144,7.686485e+00,1.944035e+00,1.626885e+01,4.114654e+00,1.761444e+01,4.454973e+00,2.359891e+01,5.968544e+00


### 0.2 Valor economico y dano historico por municipio (EUR/m2 + indemnizacion CNIH)

Dos fuentes ya calculadas en Notebook 1, ninguna con `Mun_Code` todavia:

- `valor_vivienda.csv` solo trae nombre de municipio (`municipio`), en formato INE crudo -
  algunos invertidos (`"Linea de la Concepcion (La)"`), otros con guion/espacio distinto
  (`"Velez Malaga"` vs `Velez-Malaga`). Hace falta normalizar antes de cruzar - cruzar por
  texto tal cual es justo el error que ya habiamos anotado como riesgo en Fase 1.
- `cnih_events_by_municipio.csv` ya trae `cod_municipio` (INE), ese cruce es directo.

**Cobertura real de EUR/m2: solo municipios >25.000 hab** (la fuente del Ministerio no cubre
menos) - la mayoria de los 62 son pueblos pequenos del interior, así que va a salir bastante
NaN. Eso no es un bug, es la cobertura real de la fuente - la tabla provincial de respaldo
(Fase 1, "tabla 1") no se ha descargado todavia; se decide más abajo si hace falta.

In [3]:
import re
import unicodedata


def normalize_name(name):
    # Handles the three real mismatches found between valor_vivienda.csv's INE-style
    # names and our Mun_Name: accents, "Nombre (El/La/Los/Las)" inversion, and
    # hyphen vs space ("Velez-Malaga" vs "Velez Malaga").
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.lower().strip()
    m = re.match(r"^(.*)\s*\((el|la|los|las)\)$", name)
    if m:
        name = f"{m.group(2)} {m.group(1)}".strip()
    name = name.replace("-", " ")
    name = re.sub(r"\s+", " ", name)
    return name


valor_vivienda = pd.read_csv(PROCESSED_DIR / "valor_vivienda.csv")
valor_vivienda["name_norm"] = valor_vivienda["municipio"].map(normalize_name)

admin_municipios["name_norm"] = admin_municipios["Mun_Name"].map(normalize_name)
name_to_code = dict(zip(admin_municipios["name_norm"], admin_municipios["Mun_Code"]))
valor_vivienda["Mun_Code"] = valor_vivienda["name_norm"].map(name_to_code)

matched = valor_vivienda[valor_vivienda["Mun_Code"].notna()]
unmatched = valor_vivienda[valor_vivienda["Mun_Code"].isna()]
print(f"{len(matched)} de {len(valor_vivienda)} filas de valor_vivienda emparejadas a nuestros 62 municipios")
print("Sin match (fuera de las 3 comarcas - esperado):", unmatched["municipio"].tolist())

cnih = pd.read_csv(PROCESSED_DIR / "cnih_events_by_municipio.csv")
cnih["Mun_Code"] = cnih["cod_municipio"].astype(str).str.zfill(5)

valor_economico = (
    admin_municipios[["Mun_Code", "Mun_Name"]]
    .merge(matched[["Mun_Code", "valor_eur_m2"]], on="Mun_Code", how="left")
    .merge(cnih[["Mun_Code", "n_eventos", "indemnizacion_total_eur"]], on="Mun_Code", how="left")
)
valor_economico["n_eventos"] = valor_economico["n_eventos"].fillna(0).astype(int)
valor_economico["indemnizacion_total_eur"] = valor_economico["indemnizacion_total_eur"].fillna(0)

sin_dato = valor_economico["valor_eur_m2"].isna().sum()
print(f"\n{len(valor_economico) - sin_dato} de {len(valor_economico)} municipios con EUR/m2 (cobertura >25.000 hab)")
print(f"{sin_dato} municipios sin EUR/m2 - fallback provincial no descargado todavia")

valor_economico = valor_economico.sort_values("valor_eur_m2", ascending=False, na_position="last").reset_index(drop=True)
valor_economico.to_csv(PROCESSED_DIR / "valor_economico_by_municipio.csv", index=False)

print("\n--- Ordenado por EUR/m2 (de mayor a menor, sin dato al final) ---")
display(valor_economico)

print("\n--- Mismos municipios, ordenado por indemnizacion CNIH historica (dano real documentado) ---")
display(valor_economico.sort_values("indemnizacion_total_eur", ascending=False).reset_index(drop=True))

13 de 24 filas de valor_vivienda emparejadas a nuestros 62 municipios
Sin match (fuera de las 3 comarcas - esperado): ['Arcos de la frontera', 'Cádiz', 'Chiclana de la Frontera', 'Jerez de la Frontera', 'Puerto de Santa María', 'Puerto Real', 'Rota', 'San Fernando', 'Sanlúcar de Barrameda', 'Antequera', 'Ronda']

13 de 62 municipios con EUR/m2 (cobertura >25.000 hab)
49 municipios sin EUR/m2 - fallback provincial no descargado todavia

--- Ordenado por EUR/m2 (de mayor a menor, sin dato al final) ---


,Mun_Code,Mun_Name,valor_eur_m2,n_eventos,indemnizacion_total_eur
0,29069,Marbella,4332.7,5,2190056.67
1,29901,Torremolinos,3887.3,1,169158.27
2,29025,Benalmádena,3760.8,1,1085498.49
3,29054,Fuengirola,3668.5,1,66300.64
4,29051,Estepona,3514.0,2,25617.17
5,29082,Rincón de la Victoria,3343.2,2,9092561.44
6,29067,Málaga,3240.5,16,1805168.92
7,29070,Mijas,3235.3,5,2563812.88
8,29007,Alhaurín de la Torre,2724.9,1,97697.09
9,11033,San Roque,2558.3,1,669823.62



--- Mismos municipios, ordenado por indemnizacion CNIH historica (dano real documentado) ---


,Mun_Code,Mun_Name,valor_eur_m2,n_eventos,indemnizacion_total_eur
0,29082,Rincón de la Victoria,3343.2,2,9092561.44
1,29070,Mijas,3235.3,5,2563812.88
2,29069,Marbella,4332.7,5,2190056.67
3,29067,Málaga,3240.5,16,1805168.92
4,29075,Nerja,NaN,1,1552324.53
5,29025,Benalmádena,3760.8,1,1085498.49
6,11033,San Roque,2558.3,1,669823.62
7,29094,Vélez-Málaga,2558.2,2,170968.04
8,29901,Torremolinos,3887.3,1,169158.27
9,29071,Moclinejo,NaN,1,128176.96


### 0.3 Uso de suelo dentro de la zona inundada (no del municipio entero)

Enmascara CLCplus con la geometria inundada de cada municipio (`flood_geoms`, ya calculada
en 0.1 para T100) y tabula la superficie por clase - solo lo que esta dentro del polígono
inundado, el resto del municipio no entra aqui porque no responde a la pregunta.

**Leyenda verificada contra el fichero real** (`.qml` que viene con la descarga de
Copernicus, no de memoria): clases reales 1-11; los códigos 0 (relleno del propio mosaico
`gdalwarp`, no es una clase oficial), 253 ("Coastal seawater buffer"), 254 ("Outside area")
y 255 ("No data") se excluyen del cálculo de superficie.

De paso, imperviousness medio (%) dentro de la zona inundada por municipio - HRL
Imperviousness ya es un porcentaje directo (0-100), no hace falta traducir códigos.

In [4]:
CLCPLUS_LEGEND = {
    1: "Sealed",
    2: "Woody needle leaved trees",
    3: "Woody broadleaved deciduous trees",
    4: "Woody broadleaved evergreen trees",
    5: "Low-growing woody plants",
    6: "Permanent herbaceous",
    7: "Periodically herbaceous",
    8: "Lichens and mosses",
    9: "Non and sparsely vegetated",
    10: "Water",
    11: "Snow and ice",
}
# 0 = relleno del propio mosaico (no es clase oficial), 253/254/255 = buffer costero /
# fuera de area / no data - verificado contra el .qml real, no de memoria
CLCPLUS_EXCLUDE = {0, 253, 254, 255}

PERIODO_LANDUSE = "t100"
clcplus_path = PROCESSED_DIR / "clcplus_backbone_2023_25830.tif"
imperv_path = PROCESSED_DIR / "imperviousness_2024_25830.tif"

municipios_con_inundacion = flood_by_mun.loc[
    flood_by_mun[f"area_inundada_{PERIODO_LANDUSE}_km2"] > 0, "Mun_Code"
].tolist()

landuse_rows = []
imperv_rows = []

with rasterio.open(clcplus_path) as clc_src, rasterio.open(imperv_path) as imp_src:
    pixel_area_km2 = abs(clc_src.res[0] * clc_src.res[1]) / 1e6

    for mun_code in municipios_con_inundacion:
        geom = flood_geoms.get((mun_code, PERIODO_LANDUSE))
        if geom is None:
            continue

        clc_arr, _ = rio_mask(clc_src, [geom], crop=True, filled=True, nodata=0)
        vals, counts = np.unique(clc_arr[0], return_counts=True)
        for v, c in zip(vals, counts):
            v = int(v)
            if v in CLCPLUS_EXCLUDE:
                continue
            landuse_rows.append({
                "Mun_Code": mun_code,
                "clase": CLCPLUS_LEGEND.get(v, f"codigo_{v}"),
                "area_km2": c * pixel_area_km2,
            })

        imp_arr, _ = rio_mask(imp_src, [geom], crop=True, filled=True, nodata=255)
        imp_valid = imp_arr[0][imp_arr[0] <= 100]  # 255 = fuera de mascara tras el crop
        if imp_valid.size:
            imperv_rows.append({"Mun_Code": mun_code, "imperviousness_pct_mean_flood": float(imp_valid.mean())})

landuse_in_flood = pd.DataFrame(landuse_rows)
landuse_pivot = landuse_in_flood.pivot_table(
    index="Mun_Code", columns="clase", values="area_km2", aggfunc="sum", fill_value=0
)
landuse_pivot["total_km2_clasificado"] = landuse_pivot.sum(axis=1)

imperv_df = pd.DataFrame(imperv_rows).set_index("Mun_Code")

landuse_summary = (
    admin_municipios[["Mun_Code", "Mun_Name"]]
    .merge(landuse_pivot, on="Mun_Code", how="right")
    .merge(imperv_df, on="Mun_Code", how="left")
    .merge(flood_by_mun[["Mun_Code", f"area_inundada_{PERIODO_LANDUSE}_km2"]], on="Mun_Code", how="left")
    .sort_values(f"area_inundada_{PERIODO_LANDUSE}_km2", ascending=False)
    .reset_index(drop=True)
)
landuse_summary.to_csv(PROCESSED_DIR / f"landuse_in_flood_{PERIODO_LANDUSE}.csv", index=False)

print(f"{len(landuse_summary)} municipios con superficie inundada en {PERIODO_LANDUSE.upper()}, uso de suelo desglosado")
display(landuse_summary)

36 municipios con superficie inundada en T100, uso de suelo desglosado


,Mun_Code,Mun_Name,Low-growing woody plants,Non and sparsely vegetated,Periodically herbaceous,Permanent herbaceous,Sealed,Water,Woody broadleaved deciduous trees,Woody broadleaved evergreen trees,Woody needle leaved trees,total_km2_clasificado,imperviousness_pct_mean_flood,area_inundada_t100_km2
0,29067,Málaga,0.273870,0.587207,2.808544,4.202975,5.596705,0.608789,0.703310,2.800851,0.034271,17.616523,27.382289,17.614436
1,29038,Cártama,0.166061,0.372088,1.508135,2.289979,0.498682,0.546741,0.917131,7.128221,0.076136,13.503174,2.542824,13.494642
2,29094,Vélez-Málaga,0.127193,1.081793,2.801850,2.318955,1.679791,0.038767,0.114704,4.677976,0.038568,12.879597,10.085210,12.900500
3,11008,Los Barrios,0.379981,0.129491,1.771214,2.802249,1.735744,0.590005,0.279865,1.180909,0.095820,8.965280,14.673450,8.970803
4,11033,San Roque,0.509672,0.138983,1.493347,1.685586,1.079195,0.716699,0.300848,2.386898,0.063547,8.374775,9.432774,8.392951
5,29080,Pizarra,0.155370,0.297550,1.307603,1.525121,0.074537,0.184445,0.494585,3.404944,0.041965,7.486121,0.608016,7.487019
6,11035,Tarifa,0.111307,0.188642,1.358361,4.043209,0.101415,0.058751,0.060749,0.532254,0.080033,6.534719,0.843438,6.602145
7,29012,Álora,0.207426,0.072339,0.434735,0.898846,0.101215,0.252988,0.459415,2.127515,0.031374,4.585853,1.243240,4.584525
8,29070,Mijas,0.108609,0.307442,0.626075,0.973883,1.193099,0.112506,0.165761,0.995665,0.018984,4.502024,21.542900,4.503690
9,29007,Alhaurín de la Torre,0.018884,0.036569,1.332282,0.623177,0.119500,0.030375,0.140882,1.688684,0.007494,3.997847,1.724408,4.000381


### 0.4 Cultivos dentro de la zona inundada - por municipio y vulnerabilidad relativa por cultivo

Mismo enmascarado que 0.3 pero con Crop Types en vez de CLCplus, para responder la pregunta
real: no solo cuanta superficie de cada cultivo cae en zona inundable, sino que **% de TODA
la superficie de ese cultivo en las 3 comarcas** esta expuesta - eso es lo que distingue un
cultivo estructuralmente mas vulnerable (concentrado en vega/ribera) de uno que solo tiene
algo de superficie inundada porque ocupa mucha extension en general.

Denominador (superficie total de cada cultivo en las 3 comarcas) calculado aqui mismo, no
depende de la celda 0.6 - asi esta sub-seccion se puede correr sola.

In [5]:
CROPTYPES_LEGEND = {
    0: "No Cropland",
    1110: "Wheat",
    1120: "Barley",
    1130: "Maize",
    1140: "Rice",
    1150: "Other Cereals",
    1210: "Fresh Vegetables",
    1220: "Dry Pulses",
    1310: "Potatoes",
    1320: "Sugar Beet",
    1410: "Sunflower",
    1420: "Soybeans",
    1430: "Rapeseed",
    1440: "Flax cotton and hemp",
    2100: "Grapes",
    2200: "Olives",
    2310: "Fruits",
    2320: "Nuts",
    3100: "Unclassified arable crop",
    3200: "Unclassified permanent crop",
    65535: "Outside area",
}
CROP_EXCLUDE = {0, 65535}  # No Cropland, Outside area

crop_flood_rows = []
with rasterio.open(PROCESSED_DIR / "crop_types_2023_25830.tif") as crop_src:
    pixel_area_km2 = abs(crop_src.res[0] * crop_src.res[1]) / 1e6

    for mun_code in municipios_con_inundacion:
        geom = flood_geoms.get((mun_code, PERIODO_LANDUSE))
        if geom is None:
            continue
        arr, _ = rio_mask(crop_src, [geom], crop=True, filled=True, nodata=65535)
        vals, counts = np.unique(arr[0], return_counts=True)
        for v, c in zip(vals, counts):
            v = int(v)
            if v in CROP_EXCLUDE:
                continue
            crop_flood_rows.append({
                "Mun_Code": mun_code,
                "cultivo": CROPTYPES_LEGEND.get(v, f"codigo_{v}"),
                "area_km2_inundada": c * pixel_area_km2,
            })

    # denominador: superficie total de cada cultivo en las 3 comarcas completas
    arr_total, _ = rio_mask(
        crop_src, [admin_municipios.geometry.union_all()], crop=True, filled=True, nodata=65535
    )
    vals_t, counts_t = np.unique(arr_total[0], return_counts=True)
    crop_total_area = {
        CROPTYPES_LEGEND.get(int(v), f"codigo_{int(v)}"): c * pixel_area_km2
        for v, c in zip(vals_t, counts_t)
        if int(v) not in CROP_EXCLUDE
    }

crop_flood_df = pd.DataFrame(crop_flood_rows)

# --- por municipio (tabla ancha, un cultivo por columna) ---
crop_flood_by_mun = crop_flood_df.pivot_table(
    index="Mun_Code", columns="cultivo", values="area_km2_inundada", aggfunc="sum", fill_value=0
).reset_index()
crop_flood_by_mun = admin_municipios[["Mun_Code", "Mun_Name"]].merge(crop_flood_by_mun, on="Mun_Code", how="right")
crop_flood_by_mun.to_csv(PROCESSED_DIR / f"crops_in_flood_by_municipio_{PERIODO_LANDUSE}.csv", index=False)

print(f"Cultivos dentro de zona inundada por municipio ({len(crop_flood_by_mun)} municipios con algo de cultivo inundado)")
display(crop_flood_by_mun)

# --- agregado en conjunto: vulnerabilidad relativa por tipo de cultivo ---
crop_vulnerability = crop_flood_df.groupby("cultivo")["area_km2_inundada"].sum().reset_index()
crop_vulnerability["area_total_km2"] = crop_vulnerability["cultivo"].map(crop_total_area)
crop_vulnerability["pct_de_ese_cultivo_inundado"] = (
    crop_vulnerability["area_km2_inundada"] / crop_vulnerability["area_total_km2"] * 100
)
crop_vulnerability = crop_vulnerability.sort_values("pct_de_ese_cultivo_inundado", ascending=False).reset_index(drop=True)
crop_vulnerability.to_csv(PROCESSED_DIR / f"crops_in_flood_vulnerability_{PERIODO_LANDUSE}.csv", index=False)

print("\nVulnerabilidad relativa por cultivo - % de TODA la superficie de ese cultivo en las 3 comarcas que cae en zona inundable T100")
display(crop_vulnerability)

Cultivos dentro de zona inundada por municipio (34 municipios con algo de cultivo inundado)


,Mun_Code,Mun_Name,Barley,Dry Pulses,Flax cotton and hemp,Fresh Vegetables,Fruits,Grapes,Maize,Nuts,Olives,Other Cereals,Potatoes,Rapeseed,Rice,Sunflower,Unclassified arable crop,Unclassified permanent crop,Wheat
0,11004,Algeciras,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.015087,0.000000,0.000000
1,11008,Los Barrios,0.082531,0.065545,0.000000,0.055054,0.166960,0.002898,0.037968,0.007793,0.010191,0.031673,0.019384,0.000000,0.000000,0.032972,0.160665,0.031574,0.094721
2,11022,La Línea de la Concepción,0.000000,0.000000,0.000000,0.001499,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008493,0.000000,0.000000
3,11033,San Roque,0.031773,0.000000,0.038268,0.028176,0.264178,0.013589,0.000000,0.000000,0.000799,0.007893,0.000000,0.000000,0.000000,0.037868,0.071840,0.009092,0.035970
4,11035,Tarifa,0.012689,0.140982,0.000000,0.002398,0.008793,0.000000,0.000000,0.008992,0.006594,0.202830,0.000000,0.000000,0.000000,0.221214,0.105711,0.000000,0.298949
5,11903,San Martín del Tesorillo,0.000000,0.000000,0.000000,0.000000,0.033272,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003097,0.004996,0.000000
6,29005,Algarrobo,0.000000,0.000000,0.000000,0.002398,0.023780,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001099,0.000000,0.000000,0.001499,0.000000,0.000000
7,29007,Alhaurín de la Torre,0.000000,0.023480,0.000000,0.429939,1.992529,0.000000,0.000000,0.011790,0.002898,0.076136,0.000000,0.000000,0.000000,0.000000,0.344811,0.003597,0.108609
8,29008,Alhaurín el Grande,0.000000,0.000000,0.000000,0.008793,0.148276,0.000000,0.000000,0.007893,0.002098,0.000000,0.000000,0.000000,0.000000,0.000000,0.002298,0.000000,0.001099
9,29012,Álora,0.000000,0.000000,0.000000,0.005196,2.584132,0.015087,0.000000,0.082031,0.122397,0.006994,0.000000,0.000000,0.000000,0.000000,0.042364,0.003997,0.023880



Vulnerabilidad relativa por cultivo - % de TODA la superficie de ese cultivo en las 3 comarcas que cae en zona inundable T100


,cultivo,area_km2_inundada,area_total_km2,pct_de_ese_cultivo_inundado
0,Potatoes,0.092622,0.135686,68.262150
1,Fresh Vegetables,2.723416,7.283690,37.390601
2,Fruits,28.726105,247.184662,11.621314
3,Flax cotton and hemp,0.107909,1.487352,7.255139
4,Unclassified arable crop,2.908261,42.899916,6.779175
5,Maize,0.065245,1.067604,6.111371
6,Unclassified permanent crop,0.281464,4.607435,6.108906
7,Other Cereals,0.765858,18.971487,4.036888
8,Rice,0.015287,0.448824,3.406055
9,Sunflower,0.336618,15.792253,2.131536


### 0.5 Imperviousness dentro de la zona inundada - ranking propio

El % medio de imperviousness dentro de la zona inundada ya se calculo en 0.3 (columna
`imperviousness_pct_mean_flood`), pero merece su propio ranking en vez de quedar como una
columna mas en la tabla de uso de suelo - es un eje de riesgo distinto: cuanto de lo que se
inunda ya esta construido/sellado (mas expuesto, dano mas inmediato) frente a cuanto es
suelo natural/agricola (superficie inundada grande pero menos "en riesgo" en el sentido de
bienes materiales).

Depende de `landuse_summary` (celda 0.3) - correr esa primero si el kernel se reinicio.

In [6]:
imperv_ranking = landuse_summary[
    ["Mun_Code", "Mun_Name", "imperviousness_pct_mean_flood", f"area_inundada_{PERIODO_LANDUSE}_km2"]
].copy()
imperv_ranking = imperv_ranking.sort_values("imperviousness_pct_mean_flood", ascending=False).reset_index(drop=True)
imperv_ranking.to_csv(PROCESSED_DIR / f"imperviousness_in_flood_{PERIODO_LANDUSE}.csv", index=False)

print(f"{len(imperv_ranking)} municipios, ordenados por % medio de imperviousness dentro de su zona inundada T100")
display(imperv_ranking)

36 municipios, ordenados por % medio de imperviousness dentro de su zona inundada T100


,Mun_Code,Mun_Name,imperviousness_pct_mean_flood,area_inundada_t100_km2
0,29054,Fuengirola,63.408716,1.039753
1,29082,Rincón de la Victoria,41.371469,1.060855
2,29901,Torremolinos,37.894462,0.247763
3,29005,Algarrobo,34.929295,0.474453
4,29067,Málaga,27.382289,17.614436
5,11022,La Línea de la Concepción,26.512238,0.960922
6,29070,Mijas,21.542900,4.503690
7,11008,Los Barrios,14.673450,8.970803
8,29091,Torrox,14.563300,0.617654
9,29069,Marbella,11.515190,3.209057


### 0.6 Curiosidad (no alimenta el score): reparto general de usos de suelo y cultivos en las 3 comarcas

Igual que 0.3 pero sin filtrar por zona inundable - CLCplus y Crop Types sobre el extent
administrativo completo (`admin_municipios`, los 62). Solo para tener una foto de que uso de
suelo y que cultivos dominan la Costa del Sol de estudio; no entra en la seleccion del grid
ni en los carriles del score.

In [7]:
CROPTYPES_LEGEND = {
    0: "No Cropland",
    1110: "Wheat",
    1120: "Barley",
    1130: "Maize",
    1140: "Rice",
    1150: "Other Cereals",
    1210: "Fresh Vegetables",
    1220: "Dry Pulses",
    1310: "Potatoes",
    1320: "Sugar Beet",
    1410: "Sunflower",
    1420: "Soybeans",
    1430: "Rapeseed",
    1440: "Flax cotton and hemp",
    2100: "Grapes",
    2200: "Olives",
    2310: "Fruits",
    2320: "Nuts",
    3100: "Unclassified arable crop",
    3200: "Unclassified permanent crop",
    65535: "Outside area",
}
# igual que en 0.3: 0 (No Cropland) y 65535 (Outside area) no son "cultivo", se excluyen
# del reparto de cultivos - pero 0 SI cuenta para el reparto de uso de suelo general
CROPTYPES_EXCLUDE_FROM_CROP_SHARE = {0, 65535}

study_extent = [admin_municipios.geometry.union_all()]

with rasterio.open(PROCESSED_DIR / "clcplus_backbone_2023_25830.tif") as clc_src:
    pixel_area_km2 = abs(clc_src.res[0] * clc_src.res[1]) / 1e6
    clc_arr, _ = rio_mask(clc_src, study_extent, crop=True, filled=True, nodata=0)
    vals, counts = np.unique(clc_arr[0], return_counts=True)

landuse_overall = pd.DataFrame(
    [
        {"clase": CLCPLUS_LEGEND.get(int(v), f"codigo_{int(v)}"), "area_km2": c * pixel_area_km2}
        for v, c in zip(vals, counts)
        if int(v) not in CLCPLUS_EXCLUDE
    ]
).sort_values("area_km2", ascending=False).reset_index(drop=True)
landuse_overall["pct"] = landuse_overall["area_km2"] / landuse_overall["area_km2"].sum() * 100

print("--- Uso de suelo (CLCplus) en las 3 comarcas completas ---")
display(landuse_overall)

with rasterio.open(PROCESSED_DIR / "crop_types_2023_25830.tif") as crop_src:
    pixel_area_km2 = abs(crop_src.res[0] * crop_src.res[1]) / 1e6
    crop_arr, _ = rio_mask(crop_src, study_extent, crop=True, filled=True, nodata=65535)
    vals, counts = np.unique(crop_arr[0], return_counts=True)

crops_overall = pd.DataFrame(
    [
        {"cultivo": CROPTYPES_LEGEND.get(int(v), f"codigo_{int(v)}"), "area_km2": c * pixel_area_km2}
        for v, c in zip(vals, counts)
        if int(v) not in CROPTYPES_EXCLUDE_FROM_CROP_SHARE
    ]
).sort_values("area_km2", ascending=False).reset_index(drop=True)
crops_overall["pct_de_superficie_cultivada"] = crops_overall["area_km2"] / crops_overall["area_km2"].sum() * 100

print("\n--- Cultivos dominantes (Crop Types) en las 3 comarcas, solo superficie cultivada ---")
display(crops_overall)

landuse_overall.to_csv(PROCESSED_DIR / "landuse_overall_3_comarcas.csv", index=False)
crops_overall.to_csv(PROCESSED_DIR / "crops_overall_3_comarcas.csv", index=False)

--- Uso de suelo (CLCplus) en las 3 comarcas completas ---


,clase,area_km2,pct
0,Permanent herbaceous,1394.634584,27.995975
1,Woody broadleaved evergreen trees,1359.258657,27.285836
2,Low-growing woody plants,871.860186,17.501771
3,Woody needle leaved trees,466.010012,9.354712
4,Periodically herbaceous,342.785772,6.881101
5,Sealed,328.431213,6.592947
6,Non and sparsely vegetated,134.218618,2.694312
7,Woody broadleaved deciduous trees,60.342984,1.211329
8,Water,24.011959,0.482017



--- Cultivos dominantes (Crop Types) en las 3 comarcas, solo superficie cultivada ---


,cultivo,area_km2,pct_de_superficie_cultivada
0,Olives,282.830962,34.951636
1,Fruits,247.184662,30.546544
2,Nuts,82.277886,10.167722
3,Wheat,59.473014,7.349546
4,Unclassified arable crop,42.899916,5.301478
5,Dry Pulses,21.480382,2.654499
6,Other Cereals,18.971487,2.344455
7,Sunflower,15.792253,1.951572
8,Barley,13.080628,1.616476
9,Grapes,9.891803,1.222408


### 0.7 Seleccion preliminar de municipios candidatos para el grid de detalle

Combina un criterio cualitativo (municipio costero) con el top 5 de cada listado cuantitativo ya calculado. Para "costero" reutilizamos la clase 253 de CLCplus ("Coastal seawater buffer") pero esta vez enmascarando la **geometria completa del municipio** (no solo la zona inundada como en 0.3-0.5): si esa clase aparece dentro del municipio, el municipio toca el mar. No hace falta ninguna capa de costa aparte ni listarlo a mano.

La union de: costero + top 5 por % inundado T100 + top 5 por km2 inundados T100 + top 5 por EUR/m2 + top 5 por indemnizacion CNIH + top 5 por imperviousness en zona inundada T100, da un listado preliminar amplio. El numero de criterios que cumple cada municipio (columna `n_criterios`) indica cuales son los candidatos mas solidos - los que se repiten en varias listas a la vez - para hacer el corte final a mano al grid de detalle (probablemente 5-8 municipios).

In [8]:
# "Primera linea de playa" sin capa de costa aparte: reutilizamos la clase 253 de
# CLCplus ("Coastal seawater buffer"), pero enmascarando la geometria COMPLETA del
# municipio (no la zona inundada) - si aparece esa clase, el municipio toca el mar.
coastal_rows = []
with rasterio.open(clcplus_path) as clc_src:
    for _, row in admin_municipios.iterrows():
        arr, _ = rio_mask(clc_src, [row.geometry], crop=True, filled=True, nodata=254)
        vals = np.unique(arr[0])
        coastal_rows.append({"Mun_Code": row["Mun_Code"], "es_costero": bool(253 in vals)})

costero_df = pd.DataFrame(coastal_rows)
costero_df.to_csv(PROCESSED_DIR / "municipios_costeros.csv", index=False)
print(f"{costero_df['es_costero'].sum()} de {len(costero_df)} municipios detectados como costeros (tocan clase 253)")

# Top 5 de cada listado cuantitativo ya calculado en 0.1, 0.2 y 0.5
TOP_N = 5
criterios = {
    "top5_pct_inundado_t100": set(flood_by_mun.nlargest(TOP_N, "pct_inundado_t100")["Mun_Code"]),
    "top5_km2_inundado_t100": set(flood_by_mun.nlargest(TOP_N, "area_inundada_t100_km2")["Mun_Code"]),
    "top5_valor_eur_m2": set(valor_economico.nlargest(TOP_N, "valor_eur_m2")["Mun_Code"]),
    "top5_indemnizacion_cnih": set(valor_economico.nlargest(TOP_N, "indemnizacion_total_eur")["Mun_Code"]),
    "top5_imperviousness_flood": set(imperv_ranking.nlargest(TOP_N, "imperviousness_pct_mean_flood")["Mun_Code"]),
    "costero": set(costero_df.loc[costero_df["es_costero"], "Mun_Code"]),
}

candidatos = sorted(set().union(*criterios.values()))
candidatos_df = admin_municipios[admin_municipios["Mun_Code"].isin(candidatos)][["Mun_Code", "Mun_Name"]].copy()
for nombre, s in criterios.items():
    candidatos_df[nombre] = candidatos_df["Mun_Code"].isin(s)

criterio_cols = list(criterios.keys())
candidatos_df["n_criterios"] = candidatos_df[criterio_cols].sum(axis=1)
candidatos_df = candidatos_df.sort_values(["n_criterios", "Mun_Name"], ascending=[False, True]).reset_index(drop=True)
candidatos_df.to_csv(PROCESSED_DIR / "grid_candidatos_preliminar.csv", index=False)

print(f"\n{len(candidatos_df)} municipios candidatos (union de 5 criterios cuantitativos + costero), de 62 totales")
print("Ordenados por numero de criterios que cumplen - los que salen en mas listas son los candidatos mas solidos")
pd.set_option("display.max_rows", None)
display(candidatos_df)


19 de 62 municipios detectados como costeros (tocan clase 253)

22 municipios candidatos (union de 5 criterios cuantitativos + costero), de 62 totales
Ordenados por numero de criterios que cumplen - los que salen en mas listas son los candidatos mas solidos


,Mun_Code,Mun_Name,top5_pct_inundado_t100,top5_km2_inundado_t100,top5_valor_eur_m2,top5_indemnizacion_cnih,top5_imperviousness_flood,costero,n_criterios
0,29054,Fuengirola,True,False,True,False,True,True,4
1,29067,Málaga,False,True,False,True,True,True,4
2,29069,Marbella,False,False,True,True,False,True,3
3,29082,Rincón de la Victoria,False,False,False,True,True,True,3
4,29901,Torremolinos,False,False,True,False,True,True,3
5,29094,Vélez-Málaga,True,True,False,False,False,True,3
6,29005,Algarrobo,False,False,False,False,True,True,2
7,29025,Benalmádena,False,False,True,False,False,True,2
8,29038,Cártama,True,True,False,False,False,False,2
9,29051,Estepona,False,False,True,False,False,True,2


### 0.8 Decision final: municipios para el grid de detalle (cerrado)

**9 municipios**, elegidos sobre `grid_candidatos_preliminar.csv` (0.7) priorizando `n_criterios`, con dos anadidos deliberados por motivo narrativo (no solo conteo) y un extremo geografico descartado a proposito:

- **Fuengirola, Malaga, Marbella, Rincon de la Victoria, Torremolinos, Velez-Malaga** (n_criterios 3-4) - el nucleo duro cuantitativo, cubre de sobra el tramo occidental + capital + extremo oriental.
- **Nerja** (n_criterios=2: costero + top 5 indemnizacion CNIH historica) - extremo este del area de estudio, pero entra por dano historico real documentado, no solo por ser el limite del mapa.
- **Cartama** (n_criterios=2, no costero) - contraste deliberado: zona inundada grande pero rural/agricola frente al bloque anterior 100% urbano-costero: sin el, el grid solo cuenta la mitad de la historia (urbano vs. agricola).
- **San Roque** (n_criterios=2: costero + top 5 km2 inundado absoluto) - unico representante de Campo de Gibraltar en el grid, con dato cuantitativo real detras (no solo geografia).

**Tarifa descartado a proposito** pese a ser el extremo oeste del area de estudio: n_criterios=1, y ese unico criterio es "costero" (lo cumplen 20 de los 22 candidatos, no discrimina). Cero senal en superficie inundada, valor economico, dano historico o imperviousness. Meterlo solo por simetria geografica seria el mismo tipo de pick arbitrario que se descarto al principio ("hacerlo con todo no tiene sentido, demuestro mas habilidad con criterio") - solo que por geografia en vez de comodidad. El extremo oeste ya queda cubierto por San Roque (misma comarca, costero, top 5 km2 real). Tarifa sigue presente en el choropleth principal de municipio (los 62) - solo se queda sin el zoom de detalle.

In [9]:
GRID_MUNICIPIOS_FINALES = [
    "29054",  # Fuengirola
    "29067",  # Malaga
    "29069",  # Marbella
    "29082",  # Rincon de la Victoria
    "29901",  # Torremolinos
    "29094",  # Velez-Malaga
    "29075",  # Nerja
    "29038",  # Cartama
    "11033",  # San Roque
]

grid_final = admin_municipios.copy()
grid_final["Mun_Code"] = grid_final["Mun_Code"].astype(str)
grid_final = grid_final[grid_final["Mun_Code"].isin(GRID_MUNICIPIOS_FINALES)][["Mun_Code", "Mun_Name"]]
grid_final = grid_final.merge(
    candidatos_df[["Mun_Code", "n_criterios"]].assign(Mun_Code=lambda d: d["Mun_Code"].astype(str)),
    on="Mun_Code", how="left",
)
grid_final = grid_final.sort_values("n_criterios", ascending=False).reset_index(drop=True)
grid_final.to_csv(PROCESSED_DIR / "grid_municipios_final.csv", index=False)

print(f"{len(grid_final)} municipios seleccionados para el grid de detalle (decision cerrada):")
display(grid_final)


9 municipios seleccionados para el grid de detalle (decision cerrada):


,Mun_Code,Mun_Name,n_criterios
0,29054,Fuengirola,4
1,29067,Málaga,4
2,29901,Torremolinos,3
3,29069,Marbella,3
4,29082,Rincón de la Victoria,3
5,29094,Vélez-Málaga,3
6,11033,San Roque,2
7,29038,Cártama,2
8,29075,Nerja,2


## 1. Carril urbano (EUR en riesgo)

**Antes de correr esta celda:** `buildings_25830.gpkg` no traia `Mun_Code` (se perdio al concatenar los GML por municipio en `1_data_processing.ipynb`). Ya deje corregida la celda 12 de ese notebook para que etiquete cada edificio con su `Mun_Code` (viene del nombre de la carpeta `buildings_folder/<Mun_Code>/*.gml`) antes de concatenar - hay que volver a correr **esa celda** (no todo el notebook, los ZIP ya estan descargados y descomprimidos, solo se relee y regrava `buildings_25830.gpkg`).

**Que se calcula, por municipio, en T100:**
- Numero de edificios total vs. dentro de zona inundada.
- Superficie construida (m2) total vs. inundada - usando el atributo `value` de Catastro (`officialAreaReference = grossFloorArea`, ya viene sumando todas las plantas del edificio, no solo la huella en planta) en vez de la geometria del footprint, que subestimaria un edificio de 5 plantas como si fuera de 1.
- EUR de valor construido en riesgo = superficie construida inundada x EUR/m2 del municipio (`valor_economico`) - sale NaN, no 0, en los municipios sin dato de EUR/m2 (<25.000 hab., fallback provincial todavia pendiente).
- Densidad de eventos historicos CNIH (ya calculada en 0.2, se reutiliza aqui sin recalcular).

In [10]:
buildings = gpd.read_file(PROCESSED_DIR / "buildings_25830.gpkg")
assert "Mun_Code" in buildings.columns, (
    "buildings_25830.gpkg no tiene Mun_Code todavia - vuelve a correr la celda "
    "de carga de edificios en 1_data_processing.ipynb (ya corregida) antes de seguir."
)

# "value" = grossFloorArea en m2 (atributo INSPIRE oficial de Catastro) - superficie
# construida total del edificio, ya contando todas las plantas.
buildings["area_construida_m2"] = buildings["value"]

PERIODO_URBANO = "t100"
rows = []
for _, row in flood_by_mun.iterrows():
    mun_code = row["Mun_Code"]
    bldgs_mun = buildings[buildings["Mun_Code"] == mun_code]
    n_total = len(bldgs_mun)
    area_total = bldgs_mun["area_construida_m2"].sum()

    geom = flood_geoms.get((mun_code, PERIODO_URBANO))
    if geom is not None and n_total > 0:
        inundados = bldgs_mun[bldgs_mun.geometry.intersects(geom)]
    else:
        inundados = bldgs_mun.iloc[0:0]

    n_inund = len(inundados)
    area_inund = inundados["area_construida_m2"].sum()

    rows.append({
        "Mun_Code": mun_code,
        "n_edificios_total": n_total,
        "area_construida_total_m2": area_total,
        "n_edificios_inundados_t100": n_inund,
        "pct_edificios_inundados_t100": (n_inund / n_total * 100) if n_total else 0.0,
        "area_construida_inundada_t100_m2": area_inund,
        "pct_area_construida_inundada_t100": (area_inund / area_total * 100) if area_total else 0.0,
    })

carril_urbano = pd.DataFrame(rows)
carril_urbano = admin_municipios[["Mun_Code", "Mun_Name"]].merge(carril_urbano, on="Mun_Code", how="left")
carril_urbano = carril_urbano.merge(
    valor_economico[["Mun_Code", "valor_eur_m2", "n_eventos", "indemnizacion_total_eur"]],
    on="Mun_Code", how="left",
)
carril_urbano["valor_construido_en_riesgo_eur"] = (
    carril_urbano["area_construida_inundada_t100_m2"] * carril_urbano["valor_eur_m2"]
)

carril_urbano = carril_urbano.sort_values(
    "valor_construido_en_riesgo_eur", ascending=False, na_position="last"
).reset_index(drop=True)
carril_urbano.to_csv(PROCESSED_DIR / "carril_urbano_eur_en_riesgo_t100.csv", index=False)

sin_valor = carril_urbano["valor_eur_m2"].isna().sum()
print(f"{len(carril_urbano)} municipios - carril urbano (T100)")
print(f"{sin_valor} municipios sin EUR/m2 (fallback provincial pendiente) - valor_construido_en_riesgo_eur sale NaN ahi, no 0 (no confundir con 'sin riesgo')")
pd.set_option("display.max_rows", None)
display(carril_urbano)


62 municipios - carril urbano (T100)
49 municipios sin EUR/m2 (fallback provincial pendiente) - valor_construido_en_riesgo_eur sale NaN ahi, no 0 (no confundir con 'sin riesgo')


,Mun_Code,Mun_Name,n_edificios_total,area_construida_total_m2,n_edificios_inundados_t100,pct_edificios_inundados_t100,area_construida_inundada_t100_m2,pct_area_construida_inundada_t100,valor_eur_m2,n_eventos,indemnizacion_total_eur,valor_construido_en_riesgo_eur
0,29067,Málaga,270100,55804684.0,12885,4.770455,4123566.0,7.389283,3240.5,16,1805168.92,1.336242e+10
1,29069,Marbella,171441,27523735.0,3433,2.002438,2585180.0,9.392548,4332.7,5,2190056.67,1.120081e+10
2,11033,San Roque,55156,13145623.0,2814,5.101893,3562830.0,27.102785,2558.3,1,669823.62,9.114788e+09
3,29054,Fuengirola,36797,7836046.0,5040,13.696769,1312044.0,16.743700,3668.5,1,66300.64,4.813233e+09
4,29070,Mijas,111831,14106947.0,3125,2.794395,958954.0,6.797743,3235.3,5,2563812.88,3.102504e+09
5,29082,Rincón de la Victoria,38191,5143894.0,2151,5.632217,713407.0,13.869007,3343.2,2,9092561.44,2.385062e+09
6,29094,Vélez-Málaga,87697,10288104.0,3083,3.515514,744137.0,7.232985,2558.2,2,170968.04,1.903651e+09
7,29051,Estepona,75075,12505775.0,1716,2.285714,351140.0,2.807823,3514.0,2,25617.17,1.233906e+09
8,29901,Torremolinos,31713,7084083.0,495,1.560874,85759.0,1.210587,3887.3,1,169158.27,3.333710e+08
9,11022,La Línea de la Concepción,46056,5156672.0,1225,2.659805,126425.0,2.451678,1562.6,0,0.00,1.975517e+08


## 1.1 Riesgo anualizado (EAD - dano anual esperado): pondera por frecuencia, no solo T100

**Requiere haber corrido la celda de "## 1." justo antes** - reutiliza `buildings` (ya cargado con `Mun_Code`) y `flood_geoms` (ya calculado en 0.1 para los 4 periodos), no hace falta releer ni descargar nada nuevo. Lo unico nuevo es repetir el cruce edificios x zona-inundada que ya hiciste para T100, para T10/T50/T500 tambien - 4 veces el mismo bucle, no un calculo distinto.

**Por que pesar por periodo de retorno:** T10 se inunda con el 10% de probabilidad cada ano; T500, con el 0,2%. Sumar sin pesos las 4 areas trataria un metro cuadrado que se inunda cada 10 anos igual que uno que solo se inunda en una riada de 500 anos - exactamente lo que se queria evitar. El estandar del sector (seguros/modelos de catastrofe) es el **EAD (Expected Annual Damage)**: se integra la curva perdida-vs-probabilidad-de-superacion (`p = 1/T`) con la regla del trapecio.

**Convenciones de los dos extremos de la curva (quedan documentadas para el README, no son un dato mas):**
- Extremo frecuente (p=1, T=1 ano): se asume perdida 0 - no hay datos de eventos mas frecuentes que T10, y son los de menor dano esperado individual.
- Extremo raro (mas alla de T500): se asume que la perdida se queda plana en el nivel de T500 en vez de caer a 0 - convencion conservadora habitual en EAD/AAL, para no infravalorar el riesgo de un evento mas extremo que el mayor modelado.

**Dos versiones del EAD, por cobertura de dato:**
- `ead_area_construida_m2` - proxy no monetario (superficie construida esperada en riesgo cada ano), cobertura 62/62 municipios.
- `ead_valor_eur` - version monetaria (EUR esperados en riesgo cada ano), solo donde hay EUR/m2 (los mismos ~13 municipios de 0.2, cobertura limitada por esa misma fuente).

In [11]:
PERIODOS_PROB = {"t10": 0.10, "t50": 0.02, "t100": 0.01, "t500": 0.002}  # p = 1/T, prob. anual de superacion
PERIODOS_ORDEN = ["t10", "t50", "t100", "t500"]

rows = []
for _, row in admin_municipios.iterrows():
    mun_code = row["Mun_Code"]
    bldgs_mun = buildings[buildings["Mun_Code"] == mun_code]
    n_total = len(bldgs_mun)
    area_total = bldgs_mun["area_construida_m2"].sum()

    rec = {"Mun_Code": mun_code, "n_edificios_total": n_total, "area_construida_total_m2": area_total}
    for periodo in PERIODOS_ORDEN:
        geom = flood_geoms.get((mun_code, periodo))
        if geom is not None and n_total > 0:
            area_inund = bldgs_mun[bldgs_mun.geometry.intersects(geom)]["area_construida_m2"].sum()
        else:
            area_inund = 0.0
        rec[f"area_construida_inundada_{periodo}_m2"] = area_inund
    rows.append(rec)

carril_urbano_multi = pd.DataFrame(rows)
carril_urbano_multi = carril_urbano_multi.merge(
    valor_economico[["Mun_Code", "valor_eur_m2"]], on="Mun_Code", how="left"
)
for periodo in PERIODOS_ORDEN:
    carril_urbano_multi[f"valor_en_riesgo_{periodo}_eur"] = (
        carril_urbano_multi[f"area_construida_inundada_{periodo}_m2"] * carril_urbano_multi["valor_eur_m2"]
    )


def compute_ead(row, col_prefix, col_suffix):
    # Integra la curva perdida-vs-probabilidad (regla del trapecio) con las dos
    # convenciones de cola documentadas arriba: perdida 0 en p=1, perdida plana
    # al nivel de T500 mas alla de p=0.002.
    losses = [row[f"{col_prefix}_{p}_{col_suffix}"] for p in PERIODOS_ORDEN]
    probs = [PERIODOS_PROB[p] for p in PERIODOS_ORDEN]
    p_pts = [1.0] + probs
    l_pts = [0.0] + losses
    ead = sum(
        0.5 * (l_pts[i] + l_pts[i + 1]) * (p_pts[i] - p_pts[i + 1])
        for i in range(len(p_pts) - 1)
    )
    ead += l_pts[-1] * p_pts[-1]  # cola mas alla de T500
    return ead


carril_urbano_multi["ead_area_construida_m2"] = carril_urbano_multi.apply(
    lambda r: compute_ead(r, "area_construida_inundada", "m2"), axis=1
)
carril_urbano_multi["ead_valor_eur"] = carril_urbano_multi.apply(
    lambda r: compute_ead(r, "valor_en_riesgo", "eur") if pd.notna(r["valor_eur_m2"]) else float("nan"),
    axis=1,
)

carril_urbano_multi = admin_municipios[["Mun_Code", "Mun_Name"]].merge(
    carril_urbano_multi, on="Mun_Code", how="left"
)
carril_urbano_multi = carril_urbano_multi.sort_values(
    "ead_area_construida_m2", ascending=False
).reset_index(drop=True)
carril_urbano_multi.to_csv(PROCESSED_DIR / "carril_urbano_ead_multiperiodo.csv", index=False)

print(f"{len(carril_urbano_multi)} municipios - EAD (dano anual esperado), ponderando los 4 periodos de retorno")
print(f"ead_area_construida_m2: cobertura {carril_urbano_multi['ead_area_construida_m2'].notna().sum()}/62 (proxy no monetario)")
print(f"ead_valor_eur: cobertura {carril_urbano_multi['ead_valor_eur'].notna().sum()}/62 (limitado por EUR/m2 disponible)")
pd.set_option("display.max_rows", None)
display(carril_urbano_multi)


62 municipios - EAD (dano anual esperado), ponderando los 4 periodos de retorno
ead_area_construida_m2: cobertura 62/62 (proxy no monetario)
ead_valor_eur: cobertura 13/62 (limitado por EUR/m2 disponible)


,Mun_Code,Mun_Name,n_edificios_total,area_construida_total_m2,area_construida_inundada_t10_m2,area_construida_inundada_t50_m2,area_construida_inundada_t100_m2,area_construida_inundada_t500_m2,valor_eur_m2,valor_en_riesgo_t10_eur,valor_en_riesgo_t50_eur,valor_en_riesgo_t100_eur,valor_en_riesgo_t500_eur,ead_area_construida_m2,ead_valor_eur
0,11033,San Roque,55156,13145623.0,3200052.0,3538612.0,3562830.0,4618373.0,2558.3,8.186693e+09,9.052831e+09,9.114788e+09,1.181518e+10,1787038.728,4.571781e+09
1,29069,Marbella,171441,27523735.0,2149606.0,2416689.0,2585180.0,2952719.0,4332.7,9.313598e+09,1.047079e+10,1.120081e+10,1.279325e+10,1203040.879,5.212415e+09
2,29067,Málaga,270100,55804684.0,932327.0,3668244.0,4123566.0,6489528.0,3240.5,3.021206e+09,1.188694e+10,1.336242e+10,2.102932e+10,697960.472,2.261741e+09
3,29070,Mijas,111831,14106947.0,637647.0,903919.0,958954.0,1887677.0,3235.3,2.062979e+09,2.924449e+09,3.102504e+09,6.107201e+09,373080.033,1.207026e+09
4,29054,Fuengirola,36797,7836046.0,536085.0,1235312.0,1312044.0,1497691.0,3668.5,1.966628e+09,4.531742e+09,4.813233e+09,5.494279e+09,339065.232,1.243861e+09
5,29094,Vélez-Málaga,87697,10288104.0,386770.0,546545.0,744137.0,884198.0,2558.2,9.894350e+08,1.398171e+09,1.903651e+09,2.261955e+09,226114.246,5.784455e+08
6,11008,Los Barrios,33387,3657827.0,218484.0,1403380.0,1275064.0,1661782.0,NaN,NaN,NaN,NaN,NaN,191655.528,NaN
7,29051,Estepona,75075,12505775.0,160047.0,244565.0,351140.0,483432.0,3514.0,5.624052e+08,8.594014e+08,1.233906e+09,1.698780e+09,95489.307,3.355494e+08
8,29005,Algarrobo,7918,856445.0,139646.0,154747.0,221111.0,243264.0,NaN,NaN,NaN,NaN,NaN,78839.738,NaN
9,29082,Rincón de la Victoria,38191,5143894.0,54959.0,425675.0,713407.0,1002218.0,3343.2,1.837389e+08,1.423117e+09,2.385062e+09,3.350615e+09,58519.256,1.956416e+08


## 2. Carril agricola (ha en riesgo, T100)

No hace falta recalcular nada nuevo: el desglose por cultivo dentro de zona inundada T100 ya se calculo en 0.4 (`crop_flood_by_mun`) - aqui solo se agrega esa misma tabla a una sola cifra de hectareas totales por municipio (sumando todos los tipos de cultivo, km2 -> ha), para tener un numero comparable al carril urbano. Se completa a los 62 municipios (0 donde no hay cultivo inundado, en vez de omitir la fila).

In [12]:
ha_agricola_by_mun = admin_municipios[["Mun_Code", "Mun_Name"]].merge(
    crop_flood_by_mun.drop(columns="Mun_Name"), on="Mun_Code", how="left"
)
crop_cols = [c for c in ha_agricola_by_mun.columns if c not in ("Mun_Code", "Mun_Name")]
ha_agricola_by_mun[crop_cols] = ha_agricola_by_mun[crop_cols].fillna(0.0)
ha_agricola_by_mun["ha_agricola_inundada_t100"] = ha_agricola_by_mun[crop_cols].sum(axis=1) * 100  # km2 -> ha

ha_agricola_by_mun = ha_agricola_by_mun[["Mun_Code", "Mun_Name", "ha_agricola_inundada_t100"] + crop_cols]
ha_agricola_by_mun = ha_agricola_by_mun.sort_values("ha_agricola_inundada_t100", ascending=False).reset_index(drop=True)
ha_agricola_by_mun.to_csv(PROCESSED_DIR / "carril_agricola_ha_en_riesgo_t100.csv", index=False)

print(f"{len(ha_agricola_by_mun)} municipios - carril agricola (T100), ha totales + desglose por cultivo")
pd.set_option("display.max_rows", None)
display(ha_agricola_by_mun)


62 municipios - carril agricola (T100), ha totales + desglose por cultivo


,Mun_Code,Mun_Name,ha_agricola_inundada_t100,Barley,Dry Pulses,Flax cotton and hemp,Fresh Vegetables,Fruits,Grapes,Maize,Nuts,Olives,Other Cereals,Potatoes,Rapeseed,Rice,Sunflower,Unclassified arable crop,Unclassified permanent crop,Wheat
0,29038,Cártama,859.119373,0.000400,0.002997,0.007993,0.062548,7.967417,0.004796,0.000000,0.006095,0.028576,0.047860,0.000000,0.000000,0.000000,0.000200,0.391172,0.027077,0.044063
1,29094,Vélez-Málaga,723.762909,0.000000,0.049159,0.013888,1.364056,5.374092,0.002598,0.011890,0.051457,0.052656,0.003197,0.030474,0.000000,0.015287,0.022581,0.206926,0.039367,0.000000
2,29067,Málaga,524.699883,0.011990,0.069841,0.015687,0.557333,2.928544,0.000000,0.015387,0.186843,0.001599,0.101515,0.042764,0.000000,0.000000,0.002997,0.707407,0.072839,0.532254
3,29080,Pizarra,521.142867,0.010491,0.024679,0.032073,0.054554,4.136131,0.000000,0.000000,0.359099,0.124695,0.042864,0.000000,0.000000,0.000000,0.017985,0.313737,0.078734,0.016386
4,29007,Alhaurín de la Torre,299.378882,0.000000,0.023480,0.000000,0.429939,1.992529,0.000000,0.000000,0.011790,0.002898,0.076136,0.000000,0.000000,0.000000,0.000000,0.344811,0.003597,0.108609
5,29012,Álora,288.607916,0.000000,0.000000,0.000000,0.005196,2.584132,0.015087,0.000000,0.082031,0.122397,0.006994,0.000000,0.000000,0.000000,0.000000,0.042364,0.003997,0.023880
6,11035,Tarifa,100.915352,0.012689,0.140982,0.000000,0.002398,0.008793,0.000000,0.000000,0.008992,0.006594,0.202830,0.000000,0.000000,0.000000,0.221214,0.105711,0.000000,0.298949
7,29042,Coín,94.780498,0.000500,0.000000,0.000000,0.028476,0.613186,0.005895,0.000000,0.040766,0.022281,0.020583,0.000000,0.000000,0.000000,0.000000,0.206127,0.000000,0.009992
8,29070,Mijas,82.400882,0.007094,0.005995,0.000000,0.003197,0.445926,0.000000,0.000000,0.005895,0.012989,0.199133,0.000000,0.000000,0.000000,0.000000,0.139883,0.000000,0.003897
9,11008,Los Barrios,79.992902,0.082531,0.065545,0.000000,0.055054,0.166960,0.002898,0.037968,0.007793,0.010191,0.031673,0.019384,0.000000,0.000000,0.032972,0.160665,0.031574,0.094721


## 2.1 Riesgo agricola anualizado (EAD)

Mismo motivo y misma formula del trapecio que en 1.1, aplicada a hectareas de cultivo en vez de superficie construida: T10 (10%/ano) no puede pesar igual que T500 (0,2%/ano) aunque T500 inunde mas hectareas. Aqui si hace falta recalcular: 0.4 solo enmascaro el raster de cultivos contra la zona inundada T100, asi que se repite el enmascarado para T10/T50/T500 tambien (reutiliza `flood_geoms` de 0.1, mismo raster `crop_types_2023_25830.tif`, sin cultivo desagregado por tipo aqui - solo el total agricola por municipio, igual que el agregado del paso anterior).

Mismas convenciones de cola que en 1.1 (perdida 0 en p=1, perdida plana al nivel de T500 mas alla de p=0,002) - documentadas ahi, no se repiten aqui.

In [13]:
PERIODOS_ORDEN = ["t10", "t50", "t100", "t500"]
PERIODOS_PROB = {"t10": 0.10, "t50": 0.02, "t100": 0.01, "t500": 0.002}

rows = []
with rasterio.open(PROCESSED_DIR / "crop_types_2023_25830.tif") as crop_src:
    pixel_area_km2 = abs(crop_src.res[0] * crop_src.res[1]) / 1e6
    for _, row in admin_municipios.iterrows():
        mun_code = row["Mun_Code"]
        rec = {"Mun_Code": mun_code}
        for periodo in PERIODOS_ORDEN:
            geom = flood_geoms.get((mun_code, periodo))
            if geom is None:
                rec[f"ha_agricola_inundada_{periodo}"] = 0.0
                continue
            arr, _ = rio_mask(crop_src, [geom], crop=True, filled=True, nodata=65535)
            vals, counts = np.unique(arr[0], return_counts=True)
            area_km2 = sum(
                c * pixel_area_km2 for v, c in zip(vals, counts) if int(v) not in CROP_EXCLUDE
            )
            rec[f"ha_agricola_inundada_{periodo}"] = area_km2 * 100  # km2 -> ha
        rows.append(rec)

carril_agricola_multi = pd.DataFrame(rows)


def compute_ead(row, col_prefix):
    # Misma integracion por regla del trapecio que en 1.1 - ver esa celda para las
    # convenciones de cola (perdida 0 en p=1, perdida plana al nivel de T500 mas
    # alla de p=0.002).
    losses = [row[f"{col_prefix}_{p}"] for p in PERIODOS_ORDEN]
    probs = [PERIODOS_PROB[p] for p in PERIODOS_ORDEN]
    p_pts = [1.0] + probs
    l_pts = [0.0] + losses
    ead = sum(
        0.5 * (l_pts[i] + l_pts[i + 1]) * (p_pts[i] - p_pts[i + 1])
        for i in range(len(p_pts) - 1)
    )
    ead += l_pts[-1] * p_pts[-1]
    return ead


carril_agricola_multi["ead_ha_agricola"] = carril_agricola_multi.apply(
    lambda r: compute_ead(r, "ha_agricola_inundada"), axis=1
)

carril_agricola_multi = admin_municipios[["Mun_Code", "Mun_Name"]].merge(
    carril_agricola_multi, on="Mun_Code", how="left"
)
carril_agricola_multi = carril_agricola_multi.sort_values("ead_ha_agricola", ascending=False).reset_index(drop=True)
carril_agricola_multi.to_csv(PROCESSED_DIR / "carril_agricola_ead_multiperiodo.csv", index=False)

print(f"{len(carril_agricola_multi)} municipios - EAD agricola (ha/ano esperadas en riesgo), ponderando los 4 periodos")
pd.set_option("display.max_rows", None)
display(carril_agricola_multi)


62 municipios - EAD agricola (ha/ano esperadas en riesgo), ponderando los 4 periodos


,Mun_Code,Mun_Name,ha_agricola_inundada_t10,ha_agricola_inundada_t50,ha_agricola_inundada_t100,ha_agricola_inundada_t500,ead_ha_agricola
0,29038,Cártama,648.006454,804.794940,859.119373,919.188986,366.986143
1,29094,Vélez-Málaga,599.367252,698.823828,723.762909,785.840839,336.365937
2,29080,Pizarra,287.299014,473.432886,521.142867,586.957661,170.293029
3,29067,Málaga,269.663807,482.165561,524.699883,575.187534,162.006140
4,29007,Alhaurín de la Torre,200.391912,280.924361,299.378882,317.903343,115.435463
5,29012,Álora,171.675999,246.153527,288.607916,339.415299,99.832111
6,11035,Tarifa,64.096236,101.354984,100.915352,119.869454,37.595585
7,11008,Los Barrios,59.719907,87.946231,79.992902,90.554043,34.483595
8,29070,Mijas,58.460963,76.206079,82.400882,103.103517,33.435374
9,29026,Benamargosa,55.683293,58.740728,60.369362,63.526714,30.852631


## 3. MDT02 - distancia al cauce y cota relativa (solo grid de detalle, T100)

**Alcance cerrado con Juan:** solo los edificios ya inundados (T100) dentro de los **9 municipios del grid de detalle** (0.8) - hacerlo para los 62 municipios/1,7M edificios sería demasiado pesado y no aporta nada nuevo a nivel de municipio agregado (ya está el % de superficie inundada). Esto alimenta la capa de detalle, no el choropleth principal.

**Capa de cauce: `hi_tramocurso_l`, no `hi_redsecuencia_l`** - esa segunda no está descargada (el paquete completo de hidrografía nunca se llegó a bajar entero, solo los 4 layers ya en `data/processed/`) y, aunque estuviera, es el grafo de rutado de flujo (nodos/secuencia aguas abajo) - resuelve un problema de trazado que no tenemos. `hi_tramocurso_l` ya tiene la geometria real del cauce. Se excluyen los tramos `ficticio=True` (1.845 de 19.928, ~9%) - son conectores sinteticos del grafo de red para saltar lagos/embalses/tuberias, no cauce fisico real en superficie.

**Que se calcula, por edificio:**
- `dist_cauce_m` - distancia al tramo de cauce real mas cercano (punto representativo del edificio, garantizado dentro de su huella, no el centroide que podria caer fuera en un edificio en forma de U o con patio).
- `cota_edificio_m` / `cota_cauce_m` - elevacion (MDT02, VRT ya construido) en el edificio y en el punto exacto mas cercano sobre el cauce (no un punto cualquiera del cauce - el punto de minima distancia real).
- `cota_relativa_m` = `cota_edificio_m - cota_cauce_m` - si es negativa, el edificio esta mas bajo que el cauce mas cercano en ese punto (mas vulnerable en el terreno local, mas alla de estar ya dentro del poligono SNCZI); si es muy positiva, esta elevado localmente pese a caer dentro de la zona inundada (el mecanismo de inundacion ahi probablemente no es desborde directo del cauce mas cercano).

In [14]:
import shapely

GRID_MUNICIPIOS_FINALES = ["29054", "29067", "29069", "29082", "29901", "29094", "29075", "29038", "11033"]
PERIODO_MDT = "t100"
NODATA_MDT = -32767

cauce = gpd.read_file(PROCESSED_DIR / "hi_tramocurso_l_25830.gpkg")[["ficticio", "geometry"]]
n_total_cauce = len(cauce)
cauce = cauce[~cauce["ficticio"]].reset_index(drop=True)
print(f"{len(cauce)} tramos de cauce real (excluidos {n_total_cauce - len(cauce)} ficticios de {n_total_cauce} totales)")

buildings["Mun_Code"] = buildings["Mun_Code"].astype(str)
bldgs_grid = buildings[buildings["Mun_Code"].isin(GRID_MUNICIPIOS_FINALES)].copy()

# Mismo filtro "dentro de zona inundada T100" que en "## 1.", pero solo para los 9 del grid
mask = pd.Series(False, index=bldgs_grid.index)
for mc in GRID_MUNICIPIOS_FINALES:
    geom = flood_geoms.get((mc, PERIODO_MDT))
    if geom is None:
        continue
    idx = bldgs_grid.index[bldgs_grid["Mun_Code"] == mc]
    mask.loc[idx] = bldgs_grid.loc[idx, "geometry"].intersects(geom)
bldgs_grid = bldgs_grid[mask].copy()
print(f"{len(bldgs_grid)} edificios inundados (T100) dentro de los 9 municipios del grid")

# Punto representativo (garantizado dentro de la huella, a diferencia del centroide)
bldgs_grid["punto"] = bldgs_grid.geometry.representative_point()
puntos_gdf = gpd.GeoDataFrame(bldgs_grid[["Mun_Code"]], geometry=bldgs_grid["punto"], crs=buildings.crs)

# Segundo fix - el de .values (arriba en el historial) era correcto pero el kernel
# petaba de todos modos: gpd.sjoin_nearest conserva TODOS los empates exactos antes
# de deduplicar, y con cauces reales es facil que muchos puntos queden empatados a
# varios tramos (p.ej. tramos duplicados en el limite entre demarcaciones
# hidrograficas) - la tabla intermedia puede multiplicarse mucho antes del
# drop_duplicates, lo que explica el crash tras 10 min.
#
# Se sustituye sjoin_nearest por shapely.STRtree directamente: unico indice mas
# cercano por punto (sin empates que multipliquen filas), vectorizado (sin bucle
# Python de nearest_points), y el indice que devuelve es POSICIONAL sobre el array
# usado para construir el arbol - no hay dataframe que reordene nada, así que el
# bug de alineacion original queda descartado por construccion, no solo parcheado.
assert shapely.__version__.split(".")[0] >= "2", f"hace falta shapely>=2.0 (tienes {shapely.__version__})"

cauce_dedup = cauce.drop_duplicates(subset="geometry").reset_index(drop=True)
n_dup = len(cauce) - len(cauce_dedup)
if n_dup:
    print(f"{n_dup} geometrias de cauce duplicadas exactas descartadas antes del nearest-join")

cauce_geoms = cauce_dedup.geometry.values
tree = shapely.STRtree(cauce_geoms)

puntos_arr = bldgs_grid["punto"].values
nearest_idx = tree.nearest(puntos_arr)  # 1 indice posicional por punto, vectorizado

bldgs_grid["cauce_idx"] = nearest_idx
cauce_nearest_geom = cauce_geoms[nearest_idx]
bldgs_grid["dist_cauce_m"] = shapely.distance(puntos_arr, cauce_nearest_geom)

# Punto exacto mas cercano SOBRE el cauce (no solo la distancia) - para muestrear su cota
# real. shapely.shortest_line(a, b) es vectorizado: la linea mas corta entre cada par,
# su primer vertice cae en "a" (el edificio) y el segundo en "b" (el cauce).
linea_mas_corta = shapely.shortest_line(puntos_arr, cauce_nearest_geom)
coords = shapely.get_coordinates(linea_mas_corta).reshape(-1, 2, 2)
bldgs_grid["punto_cauce"] = shapely.points(coords[:, 1, :])

with rasterio.open(PROCESSED_DIR / "relief_mdt02_25830.vrt") as mdt:
    coords_bldg = [(p.x, p.y) for p in bldgs_grid["punto"]]
    coords_cauce = [(p.x, p.y) for p in bldgs_grid["punto_cauce"]]
    bldgs_grid["cota_edificio_m"] = [v[0] for v in mdt.sample(coords_bldg)]
    bldgs_grid["cota_cauce_m"] = [v[0] for v in mdt.sample(coords_cauce)]

bldgs_grid["cota_edificio_m"] = bldgs_grid["cota_edificio_m"].replace(NODATA_MDT, np.nan)
bldgs_grid["cota_cauce_m"] = bldgs_grid["cota_cauce_m"].replace(NODATA_MDT, np.nan)
bldgs_grid["cota_relativa_m"] = bldgs_grid["cota_edificio_m"] - bldgs_grid["cota_cauce_m"]

mdt_result = bldgs_grid[["Mun_Code", "gml_id", "dist_cauce_m", "cota_edificio_m", "cota_cauce_m", "cota_relativa_m"]].copy()
mdt_result["punto_x"] = bldgs_grid["punto"].x
mdt_result["punto_y"] = bldgs_grid["punto"].y
mdt_result.to_csv(PROCESSED_DIR / "mdt02_distancia_cota_grid_t100.csv", index=False)

resumen = mdt_result.merge(admin_municipios[["Mun_Code", "Mun_Name"]], on="Mun_Code", how="left")
resumen = resumen.groupby(["Mun_Code", "Mun_Name"]).agg(
    n_edificios=("dist_cauce_m", "size"),
    dist_cauce_media_m=("dist_cauce_m", "mean"),
    dist_cauce_mediana_m=("dist_cauce_m", "median"),
    cota_relativa_media_m=("cota_relativa_m", "mean"),
    cota_relativa_mediana_m=("cota_relativa_m", "median"),
    pct_cota_relativa_negativa=("cota_relativa_m", lambda s: (s < 0).mean() * 100),
).reset_index().sort_values("dist_cauce_media_m").reset_index(drop=True)
resumen.to_csv(PROCESSED_DIR / "mdt02_resumen_por_municipio_grid_t100.csv", index=False)

n_sin_dato = mdt_result["cota_relativa_m"].isna().sum()
print(f"\n{len(mdt_result)} edificios con distancia al cauce calculada; {n_sin_dato} sin cota (nodata del MDT02 en ese punto)")
display(resumen)


18083 tramos de cauce real (excluidos 1845 ficticios de 19928 totales)
31411 edificios inundados (T100) dentro de los 9 municipios del grid

31411 edificios con distancia al cauce calculada; 0 sin cota (nodata del MDT02 en ese punto)


,Mun_Code,Mun_Name,n_edificios,dist_cauce_media_m,dist_cauce_mediana_m,cota_relativa_media_m,cota_relativa_mediana_m,pct_cota_relativa_negativa
0,29075,Nerja,217,93.281496,69.367946,2.323631,1.699997,7.373272
1,29038,Cártama,1293,112.090533,86.251617,0.472806,0.570002,35.266821
2,29082,Rincón de la Victoria,2151,155.259134,107.494561,-0.649406,0.390000,42.724314
3,29901,Torremolinos,495,185.537990,99.544770,0.544915,0.530000,21.010101
4,29069,Marbella,3433,214.746552,100.188723,1.651918,2.248001,20.244684
5,11033,San Roque,2814,284.471600,277.016887,2.701698,2.520500,4.797441
6,29067,Málaga,12885,408.597940,347.517734,-3.539758,0.273000,45.913853
7,29054,Fuengirola,5040,455.194889,472.832966,0.607616,0.965000,24.246032
8,29094,Vélez-Málaga,3083,1015.283328,1194.944570,-2.219860,-1.290001,60.006487


## 2.2 EAD desagregado por tipo de cultivo

2.1 daba el EAD agregado (todos los cultivos juntos) por municipio. Aqui se repite el mismo enmascarado del raster de cultivos para los 4 periodos, pero sin colapsar por tipo de cultivo - mismo coste que 2.1 (el desglose por cultivo ya sale gratis de `np.unique(arr, return_counts=True)`, no hay que volver a leer el raster una vez mas por cultivo). Da una fila por (municipio, cultivo) con su EAD propio, mas un ranking agregado de que cultivo concentra mas EAD en las 3 comarcas juntas (equivalente a `crop_vulnerability` de 0.4, pero ponderado por frecuencia en vez de solo T100).

In [15]:
CROPTYPES_LEGEND = {
    0: "No Cropland",
    1110: "Wheat",
    1120: "Barley",
    1130: "Maize",
    1140: "Rice",
    1150: "Other Cereals",
    1210: "Fresh Vegetables",
    1220: "Dry Pulses",
    1310: "Potatoes",
    1320: "Sugar Beet",
    1410: "Sunflower",
    1420: "Soybeans",
    1430: "Rapeseed",
    1440: "Flax cotton and hemp",
    2100: "Grapes",
    2200: "Olives",
    2310: "Fruits",
    2320: "Nuts",
    3100: "Unclassified arable crop",
    3200: "Unclassified permanent crop",
    65535: "Outside area",
}
CROP_EXCLUDE = {0, 65535}
PERIODOS_ORDEN = ["t10", "t50", "t100", "t500"]
PERIODOS_PROB = {"t10": 0.10, "t50": 0.02, "t100": 0.01, "t500": 0.002}

cultivo_rows = []
with rasterio.open(PROCESSED_DIR / "crop_types_2023_25830.tif") as crop_src:
    pixel_area_km2 = abs(crop_src.res[0] * crop_src.res[1]) / 1e6
    for _, row in admin_municipios.iterrows():
        mun_code = row["Mun_Code"]
        for periodo in PERIODOS_ORDEN:
            geom = flood_geoms.get((mun_code, periodo))
            if geom is None:
                continue
            arr, _ = rio_mask(crop_src, [geom], crop=True, filled=True, nodata=65535)
            vals, counts = np.unique(arr[0], return_counts=True)
            for v, c in zip(vals, counts):
                v = int(v)
                if v in CROP_EXCLUDE:
                    continue
                cultivo_rows.append({
                    "Mun_Code": mun_code,
                    "cultivo": CROPTYPES_LEGEND.get(v, f"codigo_{v}"),
                    "periodo": periodo,
                    "ha_inundada": c * pixel_area_km2 * 100,
                })

cultivo_df = pd.DataFrame(cultivo_rows)
cultivo_wide = cultivo_df.pivot_table(
    index=["Mun_Code", "cultivo"], columns="periodo", values="ha_inundada", fill_value=0.0
).reset_index()
for p in PERIODOS_ORDEN:
    if p not in cultivo_wide.columns:
        cultivo_wide[p] = 0.0
cultivo_wide = cultivo_wide[["Mun_Code", "cultivo"] + PERIODOS_ORDEN]


def compute_ead_cultivo(row):
    losses = [row[p] for p in PERIODOS_ORDEN]
    probs = [PERIODOS_PROB[p] for p in PERIODOS_ORDEN]
    p_pts = [1.0] + probs
    l_pts = [0.0] + losses
    ead = sum(
        0.5 * (l_pts[i] + l_pts[i + 1]) * (p_pts[i] - p_pts[i + 1])
        for i in range(len(p_pts) - 1)
    )
    ead += l_pts[-1] * p_pts[-1]
    return ead


cultivo_wide["ead_ha"] = cultivo_wide.apply(compute_ead_cultivo, axis=1)
cultivo_wide = cultivo_wide.rename(columns={p: f"ha_inundada_{p}" for p in PERIODOS_ORDEN})
cultivo_wide = admin_municipios[["Mun_Code", "Mun_Name"]].merge(cultivo_wide, on="Mun_Code", how="right")
cultivo_wide = cultivo_wide.sort_values(["Mun_Code", "ead_ha"], ascending=[True, False]).reset_index(drop=True)
cultivo_wide.to_csv(PROCESSED_DIR / "carril_agricola_ead_por_cultivo.csv", index=False)

print(f"{cultivo_wide['Mun_Code'].nunique()} municipios con algo de cultivo inundado, {len(cultivo_wide)} filas municipio x cultivo")
pd.set_option("display.max_rows", None)
display(cultivo_wide)

ead_por_cultivo_total = (
    cultivo_wide.groupby("cultivo")["ead_ha"].sum().sort_values(ascending=False).reset_index()
)
ead_por_cultivo_total.to_csv(PROCESSED_DIR / "ead_por_cultivo_total_3_comarcas.csv", index=False)
print("\nEAD total por tipo de cultivo (sumado en las 3 comarcas) - que cultivo concentra mas riesgo anualizado")
display(ead_por_cultivo_total)


34 municipios con algo de cultivo inundado, 213 filas municipio x cultivo


,Mun_Code,Mun_Name,cultivo,ha_inundada_t10,ha_inundada_t50,ha_inundada_t100,ha_inundada_t500,ead_ha
0,11004,Algeciras,Unclassified arable crop,1.498743,1.518726,1.508734,1.518726,0.825418
1,11008,Los Barrios,Fruits,14.807579,17.765099,16.695995,18.804227,8.318233
2,11008,Los Barrios,Unclassified arable crop,7.163991,21.212207,16.066523,21.591889,4.739055
3,11008,Los Barrios,Barley,8.253077,8.253077,8.253077,8.253077,4.539193
4,11008,Los Barrios,Wheat,7.723522,9.761812,9.472055,10.011602,4.369125
5,11008,Los Barrios,Fresh Vegetables,5.255592,5.515374,5.505382,6.164829,2.909969
6,11008,Los Barrios,Dry Pulses,4.766002,6.934184,6.554502,7.054083,2.748694
7,11008,Los Barrios,Maize,3.287243,3.806807,3.796815,3.806807,1.839067
8,11008,Los Barrios,Sunflower,3.297234,3.447109,3.297234,3.437117,1.821062
9,11008,Los Barrios,Unclassified permanent crop,1.348869,3.656933,3.157352,3.706891,0.876165



EAD total por tipo de cultivo (sumado en las 3 comarcas) - que cultivo concentra mas riesgo anualizado


,cultivo,ead_ha
0,Fruits,1136.846756
1,Fresh Vegetables,108.046051
2,Unclassified arable crop,97.659543
3,Wheat,38.663239
4,Other Cereals,30.601262
5,Nuts,22.110584
6,Dry Pulses,13.916167
7,Olives,13.212827
8,Sunflower,12.211347
9,Unclassified permanent crop,9.272522


## 2.3 MDT02 para el carril agricola (distancia al cauce y cota relativa, solo grid, T100)

Mismo patron que "## 3." (edificios), pero el punto de muestreo aqui es el centro de cada pixel de cultivo inundado (10m, raster `crop_types_2023_25830.tif`) en vez del punto representativo de un edificio - no hay geometria vectorial de parcela agricola en ningun dato usado, asi que el pixel es la unidad natural. Son bastantes mas puntos que edificios (los 9 municipios del grid tienen mas ha de cultivo inundado que edificios individuales en algunos casos, ej. Cartama), asi que esta celda tarda mas que la de edificios - sigue acotada a los 9 municipios del grid, no a los 62.

Mismas columnas y misma interpretacion que en edificios: `dist_cauce_m`, `cota_suelo_m`/`cota_cauce_m` (MDT02), `cota_relativa_m` = suelo - cauce (negativo = el pixel de cultivo esta mas bajo que el cauce mas cercano).

In [16]:
import shapely

GRID_MUNICIPIOS_FINALES = ["29054", "29067", "29069", "29082", "29901", "29094", "29075", "29038", "11033"]
PERIODO_MDT = "t100"
NODATA_MDT = -32767

CROPTYPES_LEGEND = {
    0: "No Cropland", 1110: "Wheat", 1120: "Barley", 1130: "Maize", 1140: "Rice",
    1150: "Other Cereals", 1210: "Fresh Vegetables", 1220: "Dry Pulses", 1310: "Potatoes",
    1320: "Sugar Beet", 1410: "Sunflower", 1420: "Soybeans", 1430: "Rapeseed",
    1440: "Flax cotton and hemp", 2100: "Grapes", 2200: "Olives", 2310: "Fruits",
    2320: "Nuts", 3100: "Unclassified arable crop", 3200: "Unclassified permanent crop",
    65535: "Outside area",
}
CROP_EXCLUDE = {0, 65535}

cauce = gpd.read_file(PROCESSED_DIR / "hi_tramocurso_l_25830.gpkg")[["ficticio", "geometry"]]
cauce = cauce[~cauce["ficticio"]].reset_index(drop=True)

pix_rows = []
with rasterio.open(PROCESSED_DIR / "crop_types_2023_25830.tif") as crop_src:
    for mc in GRID_MUNICIPIOS_FINALES:
        geom = flood_geoms.get((mc, PERIODO_MDT))
        if geom is None:
            continue
        arr, transform = rio_mask(crop_src, [geom], crop=True, filled=True, nodata=65535)
        band = arr[0]
        valid = ~np.isin(band, list(CROP_EXCLUDE))
        rr, cc = np.where(valid)
        if len(rr) == 0:
            continue
        xs, ys = rasterio.transform.xy(transform, rr, cc)
        for x, y, v in zip(xs, ys, band[rr, cc]):
            pix_rows.append({"Mun_Code": mc, "cultivo": CROPTYPES_LEGEND.get(int(v), f"codigo_{int(v)}"), "x": x, "y": y})

crop_pts = pd.DataFrame(pix_rows)
print(f"{len(crop_pts)} pixeles de cultivo inundado (T100, 10m) dentro de los 9 municipios del grid")

crop_pts_gdf = gpd.GeoDataFrame(
    crop_pts[["Mun_Code", "cultivo"]],
    geometry=gpd.points_from_xy(crop_pts["x"], crop_pts["y"]),
    crs=admin_municipios.crs,
)

# Mismo segundo fix que en la celda de edificios (ver comentario ahi con el detalle):
# sjoin_nearest conserva todos los empates exactos antes de deduplicar, lo que puede
# multiplicar la tabla intermedia y explica el crash del kernel - aqui con 221k puntos
# el riesgo es aun mayor. Se sustituye por shapely.STRtree: un unico indice mas cercano
# por punto, vectorizado, indice posicional sin ambiguedad de alineacion.
assert shapely.__version__.split(".")[0] >= "2", f"hace falta shapely>=2.0 (tienes {shapely.__version__})"

cauce_dedup = cauce.drop_duplicates(subset="geometry").reset_index(drop=True)
n_dup = len(cauce) - len(cauce_dedup)
if n_dup:
    print(f"{n_dup} geometrias de cauce duplicadas exactas descartadas antes del nearest-join")

cauce_geoms = cauce_dedup.geometry.values
tree = shapely.STRtree(cauce_geoms)

puntos_arr = crop_pts_gdf.geometry.values
nearest_idx = tree.nearest(puntos_arr)

crop_pts["cauce_idx"] = nearest_idx
cauce_nearest_geom = cauce_geoms[nearest_idx]
crop_pts["dist_cauce_m"] = shapely.distance(puntos_arr, cauce_nearest_geom)

linea_mas_corta = shapely.shortest_line(puntos_arr, cauce_nearest_geom)
coords = shapely.get_coordinates(linea_mas_corta).reshape(-1, 2, 2)
crop_pts["punto_cauce"] = shapely.points(coords[:, 1, :])

with rasterio.open(PROCESSED_DIR / "relief_mdt02_25830.vrt") as mdt:
    coords_pt = list(zip(crop_pts["x"], crop_pts["y"]))
    coords_cauce = [(p.x, p.y) for p in crop_pts["punto_cauce"]]
    crop_pts["cota_suelo_m"] = [v[0] for v in mdt.sample(coords_pt)]
    crop_pts["cota_cauce_m"] = [v[0] for v in mdt.sample(coords_cauce)]

crop_pts["cota_suelo_m"] = crop_pts["cota_suelo_m"].replace(NODATA_MDT, np.nan)
crop_pts["cota_cauce_m"] = crop_pts["cota_cauce_m"].replace(NODATA_MDT, np.nan)
crop_pts["cota_relativa_m"] = crop_pts["cota_suelo_m"] - crop_pts["cota_cauce_m"]

mdt_agricola = crop_pts[
    ["Mun_Code", "cultivo", "x", "y", "dist_cauce_m", "cota_suelo_m", "cota_cauce_m", "cota_relativa_m"]
]
mdt_agricola.to_csv(PROCESSED_DIR / "mdt02_distancia_cota_agricola_grid_t100.csv", index=False)

resumen_agricola = mdt_agricola.merge(admin_municipios[["Mun_Code", "Mun_Name"]], on="Mun_Code", how="left")
resumen_agricola = resumen_agricola.groupby(["Mun_Code", "Mun_Name"]).agg(
    n_pixeles=("dist_cauce_m", "size"),
    dist_cauce_media_m=("dist_cauce_m", "mean"),
    dist_cauce_mediana_m=("dist_cauce_m", "median"),
    cota_relativa_media_m=("cota_relativa_m", "mean"),
    cota_relativa_mediana_m=("cota_relativa_m", "median"),
    pct_cota_relativa_negativa=("cota_relativa_m", lambda s: (s < 0).mean() * 100),
).reset_index().sort_values("dist_cauce_media_m").reset_index(drop=True)
resumen_agricola.to_csv(PROCESSED_DIR / "mdt02_resumen_agricola_por_municipio_grid_t100.csv", index=False)

n_sin_dato = mdt_agricola["cota_relativa_m"].isna().sum()
print(f"\n{len(mdt_agricola)} pixeles con distancia calculada; {n_sin_dato} sin cota (nodata del MDT02)")
display(resumen_agricola)


221428 pixeles de cultivo inundado (T100, 10m) dentro de los 9 municipios del grid

221428 pixeles con distancia calculada; 0 sin cota (nodata del MDT02)


,Mun_Code,Mun_Name,n_pixeles,dist_cauce_media_m,dist_cauce_mediana_m,cota_relativa_media_m,cota_relativa_mediana_m,pct_cota_relativa_negativa
0,29054,Fuengirola,71,29.086609,16.477999,-0.405915,-0.210000,57.746479
1,29075,Nerja,2486,65.134824,39.866798,0.873537,1.300000,16.009654
2,29901,Torremolinos,261,108.317999,103.484417,-0.309004,-0.210000,70.881226
3,29082,Rincón de la Victoria,1052,136.294640,98.179246,-5.140069,-1.934998,68.631179
4,29038,Cártama,85984,140.501593,114.981903,0.155423,0.040001,47.316943
5,29069,Marbella,1224,175.557106,81.326885,0.491713,0.809999,27.696078
6,11033,San Roque,5399,217.196765,163.244210,-0.403094,-0.130000,61.029820
7,29094,Vélez-Málaga,72437,477.093555,370.405692,-1.697140,-0.430000,58.802822
8,29067,Málaga,52514,482.926563,433.812229,-3.721692,0.097499,46.977949
